
### Web-Based Text Summarization Using GenAI 

In the era of information overload, extracting meaningful insights from extensive textual content is a critical need. 
This project focuses on developing a **web-based text summarization tool**, leveraging Natural Language Processing (NLP) techniques 
to generate concise summaries from long-form content. The tool is designed to provide accurate and context-aware summarization 
to aid users in consuming key information efficiently.

In [1]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import openai
from langchain_openai import ChatOpenAI
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
client = openai.OpenAI()  
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Your message here"}]
)
print(response.choices[0].message.content)

Hello! How can I assist you today?


In [3]:
# A class to represent a Webpage
# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
class Website:
    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
# Let's try one out. Change the website and add print statements to follow along.

ed = Website("https://en.wikipedia.org/wiki/Ed_Sheeran")
print(ed.title)
#print(ed.text)

Ed Sheeran - Wikipedia


In [5]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
                and provides a short summary, ignoring text that might be navigation related. \
                Respond in markdown."

In [6]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}\n"
    user_prompt += "The contents of this website is as follows; please provide a short summary of this website in markdown.\n"
    user_prompt += "If it includes news or announcements, then summarize these too.\n\n"
    #user_prompt += website.text
    return user_prompt

In [7]:
print(user_prompt_for(ed))

You are looking at a website titled Ed Sheeran - Wikipedia
The contents of this website is as follows; please provide a short summary of this website in markdown.
If it includes news or announcements, then summarize these too.




In [8]:
# See how this function creates exactly the format above
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [9]:
# Try this out, and then try for a few more websites
print(messages_for(ed))

[{'role': 'system', 'content': 'You are an assistant that analyzes the contents of a website                 and provides a short summary, ignoring text that might be navigation related.                 Respond in markdown.'}, {'role': 'user', 'content': 'You are looking at a website titled Ed Sheeran - Wikipedia\nThe contents of this website is as follows; please provide a short summary of this website in markdown.\nIf it includes news or announcements, then summarize these too.\n\n'}]


In [10]:
# And now: call the OpenAI API. You will get very familiar with this!
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [11]:
summarize("https://en.wikipedia.org/wiki/Ed_Sheeran")

'# Ed Sheeran - Wikipedia Summary\n\nEd Sheeran is a British singer-songwriter known for his unique blend of pop, folk, and acoustic music. He gained global fame with hits such as "Shape of You," "Perfect," and "Thinking Out Loud." The website covers various aspects of his life, including his early career, notable albums, chart achievements, and collaborations with other artists.\n\n## Key Highlights\n\n- **Early Life**: Sheeran was born in Halifax, West Yorkshire, and began playing guitar and writing songs at a young age.\n- **Career Milestones**: His debut album "+", released in 2011, was a commercial success and set the stage for his subsequent works.\n- **Musical Style**: Sheeran is known for his emotive songwriting and has influenced the pop music landscape with his innovative sound.\n- **Awards and Recognition**: He has received numerous awards, including Grammy Awards and Brit Awards, affirming his status in the music industry.\n\n## News and Announcements\n\nCurrently, no speci

In [12]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(summary)

In [13]:
display_summary("https://en.wikipedia.org/wiki/Ed_Sheeran")

"# Ed Sheeran - Wikipedia Summary\n\nThe Wikipedia page for Ed Sheeran provides a comprehensive overview of the British singer-songwriter's life, career, and contributions to music. Key sections include:\n\n- **Early Life**: Details about Sheeran's background, upbringing, and early influences in music.\n- **Career**: A chronicle of his rise to fame, discography, and notable achievements, including his hit singles and albums, collaborations with other artists, and awards received.\n- **Personal Life**: Insights into his relationships and personal experiences that have shaped his music.\n- **Legacy and Influence**: Discussion on Sheeran's impact on the music industry and his role in popularizing various genres.\n  \nThe page includes references to his latest news, such as recent album releases, tours, and updates regarding his music projects.\n\nOverall, the page serves as a thorough resource for fans and researchers interested in Ed Sheeran's artistic journey."

In [14]:
display_summary("https://cnn.com")

'# Summary of CNN Website\n\nThe CNN website serves as a platform for delivering breaking news, latest updates, and videos across various topics, including world events, politics, health, entertainment, and technology. The site features real-time news coverage, in-depth articles, and multimedia content designed to keep readers informed about major developments happening globally. \n\n## Key Highlights:\n- **Breaking News**: Immediate updates on significant events as they unfold.\n- **Latest News Coverage**: Articles and videos covering a wide range of topics, ensuring diverse information.\n- **Multimedia**: A variety of videos and interactive content to complement news stories.\n\nCNN aims to provide timely, reliable information and a comprehensive viewing experience for its audience.'

In [15]:
display_summary("https://anthropic.com")

'# Summary of Anthropic Website\n\nThe Anthropic website presents the organization’s commitment to advancing artificial intelligence (AI) safety and research. Key highlights include:\n\n- **Mission and Vision**: Anthropic focuses on building AI systems that are aligned with human values and ensuring that such systems are safe and beneficial.\n\n- **Research and Development**: The website outlines various projects and initiatives aimed at exploring challenges in AI alignment and safety.\n\n- **Ethical AI**: A core aspect of Anthropic’s philosophy is the ethical considerations in AI deployment, prioritizing transparency and collaboration in AI development.\n\n- **News and Announcements**: The website regularly updates visitors with the latest advancements in AI research, new partnerships, and publications. Recent announcements may include details on research findings, product updates, or major initiatives leading to safer AI technologies.\n\nOverall, the website serves as a resource for 